# LLM10 Unbounded Consumption — Upload Artifacts & Run Evaluation

**OWASP Category**: LLM10 — Unbounded Consumption | **Risk Severity**: Medium

This notebook:
1. **Uploads** all artifact files (scenarios, checks, drivers) from the local folder structure and registers them in Okareo.
2. **Runs** the full LLM10 unbounded consumption test suite against your target AI agent.

Uploading is **idempotent** — re-running will not create duplicates (scenarios return existing, checks/drivers upsert).
Registered IDs are available in-memory for the evaluation steps below.

In [ ]:
%pip install okareo python-dotenv --quiet

In [ ]:
import sys
import json
from pathlib import Path

# Add project root for owasp.common import
_nb = globals().get("__vsc_ipynb_file__", ".")
NOTEBOOK_DIR = Path(_nb).resolve().parent
CATEGORY_DIR = NOTEBOOK_DIR.parent
if str(CATEGORY_DIR.parent.parent) not in sys.path:
    sys.path.insert(0, str(CATEGORY_DIR.parent.parent))

from okareo.checks import ModelBasedCheck, CheckOutputType
from okareo.model_under_test import Driver

from owasp.common import (
    init_okareo,
    parse_artifact,
    CodeCheckFromSource,
    build_target,
    SINGLE_TURN_DRIVER_TEMPLATE,
)

okareo, OKAREO_API_KEY = init_okareo()
print(f"✓ Okareo SDK initialized (key: ...{OKAREO_API_KEY[-5:]})")
print(f"Category directory: {CATEGORY_DIR}")


---
## Part 1 — Upload Artifacts

### Upload Scenarios

Scans `scenarios/` for `.jsonl` files and uploads each via `upload_scenario_set`.

In [ ]:
scenarios_dir = CATEGORY_DIR / "scenarios"
registered_scenarios = {}  # name -> scenario object

for jsonl_path in sorted(scenarios_dir.glob("*.jsonl")):
    scenario_name = f"LLM10-{jsonl_path.stem}"
    print(f"Uploading scenario: {scenario_name} from {jsonl_path.name}")

    scenario = okareo.upload_scenario_set(
        scenario_name=scenario_name,
        file_path=str(jsonl_path),
    )
    registered_scenarios[scenario_name] = scenario
    print(f"  ✓ Registered: {scenario_name} (ID: {scenario.scenario_id})")

print(f"\nTotal scenarios uploaded: {len(registered_scenarios)}")

### Register Checks

Scans `checks/` for `.md` files, parses YAML front matter and prompt template,
and registers each via `create_or_update_check` using `ModelBasedCheck`.

In [ ]:
checks_dir = CATEGORY_DIR / "checks"
registered_checks = {}  # name -> check id

for md_path in sorted(checks_dir.glob("*.md")):
    check_data = parse_artifact(md_path)
    print(f"Registering check: {check_data['name']} from {md_path.name}")

    check_obj = ModelBasedCheck(
        prompt_template=check_data["prompt_template"],
        check_type=CheckOutputType.PASS_FAIL,
    )
    result = okareo.create_or_update_check(
        name=check_data["name"],
        description=check_data["description"],
        check=check_obj,
    )
    registered_checks[check_data["name"]] = result.id
    print(f"  ✓ Registered: {check_data['name']} (ID: {result.id})")

print(f"\nTotal checks registered: {len(registered_checks)}")


### Register Drivers

Scans `drivers/` for `.md` files, parses YAML front matter and persona prompt,
and registers each via `create_or_update_driver` using a `Driver` object.

In [ ]:
drivers_dir = CATEGORY_DIR / "drivers"
registered_drivers = {}  # name -> driver data dict

for md_path in sorted(drivers_dir.glob("*.md")):
    driver_data = parse_artifact(md_path, default_temperature=0.6)
    print(f"Registering driver: {driver_data['name']} from {md_path.name}")

    driver_obj = Driver(
        name=driver_data["name"],
        prompt_template=driver_data["prompt_template"],
        temperature=driver_data["temperature"],
    )
    result = okareo.create_or_update_driver(driver=driver_obj)
    registered_drivers[driver_data["name"]] = result
    print(f"  ✓ Registered: {driver_data['name']} (ID: {result.id})")

print(f"\nTotal drivers registered: {len(registered_drivers)}")


### Artifact Upload Summary

In [ ]:
print("=" * 60)
print("LLM10 Unbounded Consumption — Artifact Upload Summary")
print("=" * 60)
print(f"\nScenarios ({len(registered_scenarios)}):")
for name, sc in registered_scenarios.items():
    print(f"  • {name} → {sc.scenario_id}")
print(f"\nChecks ({len(registered_checks)}):")
for name, cid in registered_checks.items():
    print(f"  • {name} → {cid}")
print(f"\nDrivers ({len(registered_drivers)}):")
for name, drv in registered_drivers.items():
    print(f"  • {name} → {drv.id}")
print("\n✓ All artifacts ready. Proceeding to evaluation...")

---
## Part 2 — Run Evaluation

### Configuration

The target agent is loaded from the shared `owasp/target.env` file (copy `owasp/target.env.example` and fill in your values).
All OWASP category notebooks reference this same file so that every control evaluates the same agent.

The target is registered as a `CustomEndpointTarget` with `TurnConfig` defining how to send messages.
LLM10 scenarios use `okareo.run_simulation()` with `max_turns=10`. Scenario 1 (infinite-loop) uses `first_turn="target"`;
Scenario 2 (resource-exhaustion) uses `first_turn="driver"`. Each scenario is paired with its own driver and check.

In [ ]:
# Target loaded from owasp/target.env. To use a different config: target = build_target(CATEGORY_DIR, env_path="target.prod.env")
target = build_target(CATEGORY_DIR)
TARGET_NAME = target.name
print(f"\u2713 Target agent: {TARGET_NAME}")

MAX_TURNS = 10

SCENARIO_DRIVER_MAP = {
    "LLM10-infinite-loop":         "LLM10-loop-inducing-driver",
    "LLM10-resource-exhaustion":   "LLM10-resource-exhaustion-driver",
}

SCENARIO_FIRST_TURN = {
    "LLM10-infinite-loop":         "target",
    "LLM10-resource-exhaustion":   "driver",
}

SCENARIO_CHECK_MAP = {
    "LLM10-infinite-loop":         ["LLM10-loop-detection-check"],
    "LLM10-resource-exhaustion":   ["LLM10-resource-policy-enforcement-check"],
}

### Multi-Turn Simulations — Both LLM10 Scenarios

Each scenario runs via `okareo.run_simulation()` with `max_turns=10`.
Scenario 1 (infinite-loop) uses `first_turn="target"`; Scenario 2 (resource-exhaustion) uses `first_turn="driver"`.
Each scenario is paired with its dedicated driver and check via `SCENARIO_DRIVER_MAP` and `SCENARIO_CHECK_MAP`.

In [ ]:
simulation_results = {}  # scenario_name -> test run result

for scenario_name, driver_name in SCENARIO_DRIVER_MAP.items():
    if scenario_name not in registered_scenarios:
        print(f"⚠ Scenario {scenario_name} not found in registered scenarios — skipping")
        continue

    if driver_name not in registered_drivers:
        print(f"⚠ Driver {driver_name} not found in registered drivers — skipping {scenario_name}")
        continue

    print(f"\n{'='*60}")
    print(f"Running simulation: {scenario_name}")
    print(f"Driver: {driver_name} | Max turns: {MAX_TURNS}")
    print(f"{'='*60}")

    try:
        driver_reg = registered_drivers[driver_name]
        multi_turn_driver = Driver(
            temperature=driver_reg.temperature if hasattr(driver_reg, "temperature") else 0.7,
            name=driver_name,
            prompt_template=driver_reg.prompt_template,
        )

        scenario = registered_scenarios[scenario_name]

        simulation_run = okareo.run_simulation(
            target=target,
            driver=multi_turn_driver,
            name=f"LLM10 Simulation — {scenario_name}",
            api_key=OKAREO_API_KEY,
            first_turn=SCENARIO_FIRST_TURN.get(scenario_name, "target"),
            scenario=scenario,
            max_turns=MAX_TURNS,
            checks=SCENARIO_CHECK_MAP.get(scenario_name, []),
        )
        simulation_results[scenario_name] = simulation_run
        print(f"  ✓ Simulation complete: {simulation_run.id}")
        if hasattr(simulation_run, "app_link") and simulation_run.app_link:
            print(f"  View: {simulation_run.app_link}")
    except Exception as e:
        print(f"  ✗ Error: {e}")
        simulation_results[scenario_name] = None

### Results Summary

In [ ]:
print("\n" + "=" * 60)
print("LLM10 UNBOUNDED CONSUMPTION — EVALUATION RESULTS")
print("OWASP Category: LLM10 | Risk Severity: Medium")
print("=" * 60)

print(f"\n{'Scenario':<46} {'Status':<10} {'Link / Run ID'}")
print("-" * 110)
for name, result in simulation_results.items():
    if result is None:
        print(f"{name:<46} {'ERROR':<10} N/A")
    else:
        link = getattr(result, "app_link", None) or result.id
        print(f"{name:<46} {'COMPLETE':<10} {link}")

errors = sum(1 for r in simulation_results.values() if r is None)
print(f"\nTotal evaluated: {len(simulation_results)} | Errors: {errors}")
if not errors:
    print("✓ All scenarios completed. See Okareo dashboard for full results.")

### Detailed Results (Optional)

Retrieve per-row scores and conversation transcripts for any completed simulation run.

In [ ]:
# Uncomment to inspect a specific completed run in detail:
# RUN_ID = "<paste_run_id_here>"
# detailed = okareo.get_test_run(RUN_ID)
# for row in (detailed.model_results or []):
#     print(f"Input:   {str(row.get('scenario_input', ''))[:80]}")
#     print(f"Output:  {str(row.get('model_output', ''))[:80]}")
#     print(f"Checks:  {row.get('check_scores', {})}")
#     print("-" * 40)